# Verifikasi kandidat SENA v0.6

Ekspor hanya setelah training selesai. Pemeriksaan kontrak dan inferensi CPU memakai hash bobot/config yang eksplisit. Parameter tidak diubah selama evaluasi ini. Acuan manusia lama 50 dan 24 usulan yang sudah diketahui merupakan regresi pengembangan. Lima puluh narasi baru telah dibekukan sebelum training dan hanya diuji sekali setelah model dipilih. Acuan baru masih berlabel asisten, sehingga metriknya tidak menjadi persetujuan mutu manusia otomatis. Tidak ada training atau fitting dalam notebook ini.

In [1]:
from pathlib import Path
import os,sys,json,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
WORK=Path(os.environ['SLR09_WORK_DIR'])
RUN=WORK/'postfit_scope_v06'
OUTPUT=WORK/'quality_diagnostics_v06'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Step failed: '+str(p.returncode))
assert json.loads((RUN/'run_state.json').read_text())['status']=='SUCCEEDED'

In [2]:
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_scope_candidate_v06.py'),
 '--run',str(RUN),'--output',str(WORK/'exports')])

{
  "archive": "/workspace/scratch/e5625dd03138/silar09-next-v06/exports/SILAR_SENA_scope_v0.6_20261019.zip",
  "bytes": 411812430,
  "sha256": "b764aa579373003b736d25449ce01bd6b21a82dda5d71482fb51d5ff354db1b2",
  "model_id": "SILAR_SENA_scope_v06_20261019T053247",
  "selected_epoch": 1,
  "selected_bias": 3.4,
  "quality_acceptance_approved": false
}


In [3]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),
 '--evidence-dir',str(WORK/'contract_tests_v06')])

test_explicit_breathing_condition_is_kept (test_candidate_runtime.ActiveNormalizationTests.test_explicit_breathing_condition_is_kept) ... ok
test_literal_impact_still_maps_to_impact (test_candidate_runtime.ActiveNormalizationTests.test_literal_impact_still_maps_to_impact) ... ok
test_negated_breathing_condition_is_not_positive_victim (test_candidate_runtime.ActiveNormalizationTests.test_negated_breathing_condition_is_not_positive_victim) ... ok
test_unconscious_condition_is_not_misread_as_negation (test_candidate_runtime.ActiveNormalizationTests.test_unconscious_condition_is_not_misread_as_negation) ... ok
test_candidate_directory_has_explicit_identity_and_hashes (test_candidate_runtime.CandidateAssetTests.test_candidate_directory_has_explicit_identity_and_hashes) ... ok
test_config_labels_must_match_even_if_hash_was_updated (test_candidate_runtime.CandidateAssetTests.test_config_labels_must_match_even_if_hash_was_updated) ... ok
test_different_weights_cannot_be_accepted_as_candidate (

In [4]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_scope_diagnostics_v06.py'),
 '--previous',str(ROOT/'evidence/quality_fix_v05/previous_50'),
 '--seen',str(ROOT/'data/quality_fix_fresh_challenge_v0.5.json'),
 '--fresh',str(ROOT/'data/frozen_new_diagnostic_v0.6.json'),
 '--sena-dir',str(RUN/'best_model'),
 '--nara-archive',str(WORK/'assets/CT/SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--training-run',str(RUN),'--output',str(OUTPUT)])

Old pilot 10 / 50
Old pilot 20 / 50
Old pilot 30 / 50
Old pilot 40 / 50
Old pilot 50 / 50
seen24 10 / 24
seen24 20 / 24
new50 10 / 50
new50 20 / 50
new50 30 / 50
new50 40 / 50
new50 50 / 50
{
  "old50_targets_met": true,
  "seen24": {
    "accuracy": 0.9166666666666666,
    "macro_f1": 0.9147619047619047,
    "non_event_recall": 0.9166666666666666,
    "minimum_event_recall": 0.6666666666666666,
    "targets_met": false,
    "errors": 2
  },
  "new50": {
    "accuracy": 0.78,
    "macro_f1": 0.7940258862165555,
    "non_event_recall": 0.9,
    "minimum_event_recall": 0.6,
    "targets_met": false,
    "errors": 11
  },
  "quality_acceptance_approved": false
}
{
  "schema": "slr09-scope-diagnostics-0.6",
  "status": "SUCCEEDED",
  "started_at": "2026-10-19T05:34:08.752630+00:00",
  "training_executed": false,
  "parameter_fitting_executed": false,
  "model_inference_executed": true,
  "quality_acceptance_approved": false,
  "card_done": false,
  "new_human_reference_review_completed": f

In [5]:
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['parameter_fitting_executed'] and not state['quality_acceptance_approved']
print(json.dumps(state['diagnostic_summary'],indent=2))
print('Human acceptance remains pending; no production deployment.')

{
  "seen24": {
    "accuracy": 0.9166666666666666,
    "macro_f1": 0.9147619047619047,
    "non_event_recall": 0.9166666666666666,
    "minimum_event_recall": 0.6666666666666666,
    "targets_met": false,
    "errors": 2
  },
  "new50": {
    "accuracy": 0.78,
    "macro_f1": 0.7940258862165555,
    "non_event_recall": 0.9,
    "minimum_event_recall": 0.6,
    "targets_met": false,
    "errors": 11
  }
}
Human acceptance remains pending; no production deployment.
